> **対応するブログ記事**: [#10 大腸がんのステージ進行に伴うタンパク質変動をクラスター分析で可視化する](../blog/article-10-stage.md)
>
> このNotebookはブログ記事 #10 のコードをセルごとに実行できるインタラクティブ版です。ANOVAやクラスター分析の詳しい解説はブログ記事を参照してください。

# Step 10: ステージ別解析（Figure 3）

In [ ]:
import re  # 正規表現モジュール（サンプル名から重複サフィックスを除去するために使用）
import numpy as np  # 数値計算ライブラリ（配列操作・統計処理に使用）
import pandas as pd  # データフレーム操作ライブラリ（CSV読み込み・テーブル処理に使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリのメインモジュール
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec  # 複雑なサブプロットレイアウトを構築するためのクラス
from matplotlib.patches import Patch  # 凡例用のカラーパッチを作成するクラス
import seaborn as sns  # 統計的可視化ライブラリ（ヒートマップ描画に使用）
from scipy import stats  # SciPyの統計関数モジュール（ANOVA検定に使用）
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list  # 階層的クラスタリング関連の関数群
from statsmodels.stats.multitest import multipletests  # 多重検定補正（BH法によるFDR補正）を行う関数

# Jupyter Notebook内にグラフをインラインで表示するマジックコマンド
%matplotlib inline

In [ ]:
# --- 定数・パス設定 ---
RESULTS = "../results"  # 前処理済みデータや結果ファイルの出力先ディレクトリ
FIG_DIR = f"{RESULTS}/figures"  # 生成した図の保存先ディレクトリ
TABLE_DIR = f"{RESULTS}/tables"  # 生成した表（CSV/Excel）の保存先ディレクトリ
RAW_DIR = "../data/raw"  # 論文の補足テーブル（Supplementary Table）等の生データ格納先

N_CLUSTERS = 30  # Ward法による階層的クラスタリングで生成するクラスター数の上限
FDR_THRESHOLD = 0.01  # ANOVA後のBH-FDR補正で有意とみなすFDR閾値（1%）
STAGE_ORDER = ["Normal", "I", "II", "III", "IV"]  # 大腸がんステージの表示順序（正常→Stage I〜IV）

# 各ステージに割り当てるカラーコード（正常=青、早期=緑/黄、進行=橙/赤）
STAGE_COLORS = {
    "Normal": "#4EAED1",  # 正常組織: 水色
    "I": "#66BB6A",       # Stage I: 緑
    "II": "#FFD54F",      # Stage II: 黄
    "III": "#FFA726",     # Stage III: 橙
    "IV": "#E8524A",      # Stage IV: 赤
}

In [ ]:
# --- データ読み込み + ステージ付与 ---
# 前処理済みタンパク質発現量マトリクスを読み込む（行=タンパク質, 列=サンプル）
df = pd.read_csv(f"{RESULTS}/preprocessed_data.csv", index_col=0)
# サンプル情報（サンプル名、組織タイプ等）を読み込む
sample_info = pd.read_csv(f"{RESULTS}/sample_info.csv")

# clinical_info.csv から Sample_N→"Normal", Sample_T→Stage をマッピング
# 臨床情報ファイルを読み込む（患者ごとのステージ情報を含む）
clinical = pd.read_csv(f"{RESULTS}/clinical_info.csv")
# サンプル名→ステージの対応辞書を構築
stage_map = {}
for _, row in clinical.iterrows():
    # 正常組織サンプル（Sample_N）には "Normal" を割り当て
    stage_map[row["Sample_N"]] = "Normal"
    # 腫瘍サンプル（Sample_T）には臨床ステージ（I〜IV）を割り当て
    stage_map[row["Sample_T"]] = row["Stage"]

# sample_infoに "Stage" 列を追加（重複サフィックス _dup1 等を除去してからマッピング）
sample_info["Stage"] = sample_info["Sample"].apply(
    lambda x: stage_map.get(re.sub(r"_dup\d+$", "", x))  # 重複サンプルの末尾 _dupN を除去して検索
)

# データの形状とステージごとのサンプル数を表示
print(f"データ: {df.shape[0]} タンパク質 × {df.shape[1]} サンプル")
print(sample_info["Stage"].value_counts().to_string())  # 各ステージのサンプル数を表示

## ANOVA

In [ ]:
def run_anova(df, sample_info):
    """各タンパク質について One-way ANOVA + BH-FDR 補正を実行。"""
    # データに存在するステージのみを定義順で抽出
    stages = [s for s in STAGE_ORDER if s in sample_info["Stage"].values]

    # ステージごとのサンプル列名を事前に辞書化（検索の高速化）
    stage_samples = {
        s: [c for c in sample_info[sample_info["Stage"] == s]["Sample"] if c in df.columns]
        for s in stages
    }

    results = []  # ANOVA結果を格納するリスト
    for protein in df.index:  # 各タンパク質について反復
        groups = []  # ステージごとの発現値グループを格納
        for s in stages:
            # 該当ステージのサンプルから発現値を取得し、欠損値を除外
            vals = df.loc[protein, stage_samples[s]].dropna().values
            # サンプル数が2以上の場合のみグループに追加（ANOVA実行に最低2サンプル必要）
            if len(vals) >= 2:
                groups.append(vals)
        # グループが2つ未満の場合はANOVAを実行できないのでスキップ
        if len(groups) < 2:
            continue
        # One-way ANOVA検定を実行（F統計量とp値を取得）
        # F統計量: 群間分散/群内分散の比。大きいほどステージ間で発現差が大きい
        f_stat, p_val = stats.f_oneway(*groups)
        results.append({"Protein": protein, "F_statistic": f_stat, "P_value": p_val})

    # 結果リストをDataFrameに変換
    result_df = pd.DataFrame(results)

    # Benjamini-Hochberg法によるFDR（偽発見率）補正を実施
    # 多重検定の問題: 数千のタンパク質を同時検定すると偽陽性が増えるため補正が必要
    _, fdr, _, _ = multipletests(result_df["P_value"], method="fdr_bh")
    result_df["FDR"] = fdr  # FDR補正後のp値を列として追加
    result_df["Significant"] = fdr < FDR_THRESHOLD  # FDR閾値未満のタンパク質を有意とマーク
    return result_df  # ANOVA結果のDataFrameを返す

In [ ]:
# 全タンパク質に対してANOVA検定を実行し、ステージ間で有意に変動するタンパク質を特定
anova_df = run_anova(df, sample_info)
# ANOVA結果をCSVファイルとして保存（タンパク質名, F統計量, p値, FDR, 有意フラグ）
anova_df.to_csv(f"{TABLE_DIR}/anova_results.csv", index=False)
# 有意なタンパク質の数を集計して表示
n_sig = anova_df["Significant"].sum()
print(f"検定: {len(anova_df)} タンパク質 → 有意 (FDR<{FDR_THRESHOLD}): {n_sig}")

## クラスタリング

In [ ]:
def compute_stage_medians(df, sample_info):
    """各ステージの中央値を計算。行=タンパク質、列=ステージ。"""
    # データに存在するステージのみを定義順で抽出
    stages = [s for s in STAGE_ORDER if s in sample_info["Stage"].values]
    medians = {}  # ステージ名→中央値Seriesの辞書
    for s in stages:
        # 該当ステージに属するサンプルの列名リストを取得
        cols = [c for c in sample_info[sample_info["Stage"] == s]["Sample"] if c in df.columns]
        if cols:
            # 各タンパク質について、該当ステージの全サンプルの中央値を計算
            # 中央値を使う理由: 平均値より外れ値の影響を受けにくい
            medians[s] = df[cols].median(axis=1)
    # 辞書からDataFrameを構築（行=タンパク質, 列=ステージ）
    return pd.DataFrame(medians)

In [ ]:
# ステージ別中央値の計算（行=タンパク質, 列=ステージ）
median_df = compute_stage_medians(df, sample_info)

# ANOVAで有意と判定されたタンパク質のリストを抽出
sig_proteins = anova_df[anova_df["Significant"]]["Protein"].tolist()
# 有意なタンパク質がゼロの場合、FDR上位200個をフォールバックとして使用
if not sig_proteins:
    sig_proteins = anova_df.nsmallest(min(200, len(anova_df)), "FDR")["Protein"].tolist()

# 有意タンパク質のステージ別中央値データを抽出
sig_data = median_df.loc[median_df.index.isin(sig_proteins)]

# Z-score標準化: 各タンパク質を行方向に平均0・標準偏差1に変換
# ステージ間の相対的な変動パターンを比較可能にするための正規化
z_data = sig_data.apply(lambda x: (x - x.mean()) / x.std(), axis=1).dropna()

# 実際のクラスター数を決定（データ数が少ない場合はN_CLUSTERSより小さくなる）
actual_clusters = min(N_CLUSTERS, len(z_data))
# Ward法による階層的クラスタリングを実行（分散の増加を最小化する併合基準）
Z_linkage = linkage(z_data.values, method="ward")
# リンケージ結果から指定数のクラスターに分割（各タンパク質にクラスター番号を付与）
clusters = fcluster(Z_linkage, t=actual_clusters, criterion="maxclust")

# クラスター割り当て結果をDataFrameに変換してCSV保存
cluster_df = pd.DataFrame({"Protein": z_data.index, "Cluster": clusters})
cluster_df.to_csv(f"{TABLE_DIR}/cluster_assignments.csv", index=False)
# クラスタリング結果のサマリーを表示
print(f"{len(z_data)} タンパク質 → {actual_clusters} クラスター")

## Figure 3

In [ ]:
def plot_figure3(df, sample_info, z_data, clusters):
    """Figure 3: 2x2 パネル（ヒートマップ + ラインプロット × 4 クラスター）。"""

    # --- 論文補足 Table S9-S12 からクラスター読み込み（なければ自前） ---
    # 各タプル: (ファイル名, 論文中のクラスター番号, 発現変動の方向)
    cluster_files = [
        ("Supplementary Table S9 Fig3A_Clustre3_34 protein.xlsx", 3, "Increased"),    # Fig3A: 増加クラスター3 (34タンパク質)
        ("Supplementary Table S10 Fig3B_Clustre14_1324 protein.xlsx", 14, "Increased"),  # Fig3B: 増加クラスター14 (1324タンパク質)
        ("Supplementary Table S11 Fig3C_Clustre20_16 protein.xlsx", 20, "Decreased"),   # Fig3C: 減少クラスター20 (16タンパク質)
        ("Supplementary Table S12 Fig3D_Clustre25_1062 protein.xlsx", 25, "Decreased"),  # Fig3D: 減少クラスター25 (1062タンパク質)
    ]
    selected = []  # 描画対象として選択されたクラスター情報のリスト
    for fname, cl_num, direction in cluster_files:
        try:
            # 論文の補足テーブルExcelファイルを読み込む
            cdf = pd.read_excel(f"{RAW_DIR}/{fname}", header=0)
            # 遺伝子名/シンボル列を自動検出（"gene"または"symbol"を含む列名を探す）
            gene_col = [c for c in cdf.columns if "gene" in c.lower() or "symbol" in c.lower()]
            # 遺伝子名列が見つかればそこから、なければ6列目（インデックス5）からタンパク質名を取得
            genes = cdf[gene_col[0]].dropna().tolist() if gene_col else cdf.iloc[:, 5].dropna().tolist()
            # データフレームのインデックスに存在するタンパク質のみを抽出（名前の不一致を除外）
            matched = [g for g in genes if g in df.index]
            # クラスター情報を辞書としてリストに追加
            selected.append({"cluster": cl_num, "n": len(matched), "genes": matched, "direction": direction})
        except Exception:
            pass  # ファイルが見つからない場合はスキップ

    # 4つのクラスターすべてが読み込めなかった場合のフォールバック処理
    if len(selected) != 4:
        # フォールバック: 自前クラスタリングから増加2 + 減少2 を選択
        print("論文クラスター不完全 → 自前クラスタリングを使用")
        infos = []  # 各クラスターの情報を格納するリスト
        for cl in np.unique(clusters):  # 全クラスター番号をループ
            mask = clusters == cl  # 該当クラスターに属するタンパク質のブールマスク
            if mask.sum() < 5:  # タンパク質数が5未満のクラスターは除外
                continue
            # クラスター内の全タンパク質の平均Z-scoreプロファイルを計算
            profile = z_data.iloc[mask].mean(axis=0)
            # トレンド: 最終ステージと最初のステージのZ-score差（正=増加、負=減少）
            trend = profile.iloc[-1] - profile.iloc[0]
            infos.append({"cluster": cl, "n": int(mask.sum()), "genes": z_data.index[mask].tolist(),
                          "direction": "Increased" if trend > 0 else "Decreased", "trend": trend})
        # 増加トレンドが最も強い上位2クラスターを選択
        inc = sorted([c for c in infos if c["direction"] == "Increased"], key=lambda x: x["trend"], reverse=True)[:2]
        # 減少トレンドが最も強い上位2クラスターを選択
        dec = sorted([c for c in infos if c["direction"] == "Decreased"], key=lambda x: x["trend"])[:2]
        # 増加2つ + 減少2つ の計4クラスターを描画対象とする
        selected = inc + dec

    # --- Figure 描画 ---
    panel_labels = ["(a)", "(b)", "(c)", "(d)"]  # 各パネルのラベル
    # Figure全体のサイズを設定（幅16インチ × 高さ20インチ）
    fig = plt.figure(figsize=(16, 20))
    # 外側のGridSpec: 2行2列のグリッドレイアウトを定義（4パネル分）
    # hspace=0.4: 行間の垂直スペース、wspace=0.3: 列間の水平スペース
    outer_gs = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

    # 4つのクラスターパネルを順に描画
    for idx, info in enumerate(selected):
        # 内側のGridSpec: 各パネルを上下2段に分割（上=ヒートマップ、下=ラインプロット）
        # height_ratios=[3,1]: ヒートマップを3、ラインプロットを1の高さ比率に設定
        inner_gs = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer_gs[idx],
                                          height_ratios=[3, 1], hspace=0.15)
        # ヒートマップ用のAxesオブジェクトを作成
        ax_heat = fig.add_subplot(inner_gs[0])
        # ラインプロット用のAxesオブジェクトを作成
        ax_line = fig.add_subplot(inner_gs[1])

        # データフレームに存在するタンパク質のみをフィルタリング
        proteins = [g for g in info["genes"] if g in df.index]

        # 各ステージの中央値を計算してステージ別中央値行列を構築
        stage_med = {}
        for s in STAGE_ORDER:
            # 該当ステージのサンプル列名を取得
            cols = [c for c in sample_info[sample_info["Stage"] == s]["Sample"] if c in df.columns]
            if cols:
                # 対象タンパク質の該当ステージにおける中央値を計算
                stage_med[s] = df.loc[proteins, cols].median(axis=1)
        # ステージ別中央値のDataFrameを構築（行=タンパク質, 列=ステージ）
        med_matrix = pd.DataFrame(stage_med)

        # 行方向（各タンパク質）でZ-score標準化を実行
        # 標準偏差が0のタンパク質（全ステージで同一値）はゼロ埋め
        cz = med_matrix.apply(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else x * 0, axis=1)

        # タンパク質をWard法クラスタリング順にソート（ヒートマップの視覚的な一貫性のため）
        if len(cz) > 1:
            # leaves_list: デンドログラムの葉ノード順序を取得し、その順番で行を並べ替え
            cz = cz.iloc[leaves_list(linkage(cz.values, method="ward"))]

        # ヒートマップを描画（赤=高発現、緑=低発現、中央=0）
        sns.heatmap(cz, cmap="RdYlGn_r", center=0, vmin=-2, vmax=2, ax=ax_heat,
                    xticklabels=True, yticklabels=False,  # Y軸ラベルは非表示（タンパク質数が多いため）
                    cbar_kws={"shrink": 0.5, "label": "Z-score"})  # カラーバーを50%に縮小
        # X軸ラベルを "Non-tumor" / "Stage I〜IV" に変更（45度回転で読みやすく）
        ax_heat.set_xticklabels(
            ["Non-tumor" if s == "Normal" else f"Stage {s}" for s in cz.columns],
            fontsize=7, rotation=45, ha="right")

        # ステージカラーバー: ヒートマップ上部にステージの色帯を追加
        hp = ax_heat.get_position()  # ヒートマップの位置情報を取得
        # ヒートマップ直上にカラーバー用の新しいAxesを追加（幅85%・高さ4%の薄い帯）
        ax_bar = fig.add_axes([hp.x0, hp.y1 + 0.005, hp.width * 0.85, hp.height * 0.04])
        # 各ステージの色を帯として描画
        colors_list = [STAGE_COLORS.get(s, "#888") for s in cz.columns]
        for i, c in enumerate(colors_list):
            # axvspan: 垂直方向の色帯を描画（各ステージに対応する色）
            ax_bar.axvspan(i, i + 1, facecolor=c, edgecolor="white", linewidth=0.5)
        ax_bar.set_xlim(0, len(colors_list))  # X軸範囲をステージ数に合わせる
        ax_bar.set_xticks([]); ax_bar.set_yticks([])  # カラーバーの目盛りを非表示

        # パネルタイトル: クラスター番号・タンパク質数・変動方向を表示
        ax_heat.set_title(
            f"{panel_labels[idx]} Cluster {info['cluster']}: {info['n']} proteins ({info['direction']})",
            fontsize=11, fontweight="bold", pad=15)

        # --- ラインプロット: クラスター内の全タンパク質の発現プロファイル ---
        x_pos = range(len(cz.columns))  # X軸の位置（ステージ数分）
        # X軸ラベル: "Normal"を"Non-tumor"に変換
        labels = ["Non-tumor" if s == "Normal" else s for s in cz.columns]
        # 個々のタンパク質のZ-scoreプロファイルを灰色の細い線で描画（背景パターン）
        for _, row in cz.iterrows():
            ax_line.plot(x_pos, row.values, color="gray", alpha=0.15, linewidth=0.3)
        # クラスター全体の平均Z-scoreプロファイルを黒い太線で描画
        mean_prof = cz.mean(axis=0)
        ax_line.plot(x_pos, mean_prof.values, "k-", linewidth=2.5)
        # 各ステージ位置にステージ色の丸マーカーを描画
        for i, (xp, ym) in enumerate(zip(x_pos, mean_prof.values)):
            ax_line.plot(xp, ym, "o", color=colors_list[i], markersize=8,
                         markeredgecolor="black", markeredgewidth=0.5, zorder=5)
        ax_line.set_xticks(list(x_pos))  # X軸の目盛り位置を設定
        ax_line.set_xticklabels(labels, fontsize=8)  # X軸ラベルを設定
        ax_line.set_ylabel("Z-score", fontsize=8)  # Y軸ラベル
        ax_line.axhline(0, color="gray", ls="--", lw=0.5)  # Z-score=0の基準線（破線）
        ax_line.spines["top"].set_visible(False)  # 上枠線を非表示（見やすさのため）
        ax_line.spines["right"].set_visible(False)  # 右枠線を非表示

    # Figure全体の凡例を下部中央に配置（5ステージ分のカラーパッチ）
    fig.legend(
        handles=[Patch(fc=STAGE_COLORS[s], label="Non-tumor" if s == "Normal" else f"Stage {s}")
                 for s in STAGE_ORDER],
        loc="lower center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.01))

    # 図をPNGファイルとして保存（150dpi, 余白を自動調整）
    path = f"{FIG_DIR}/fig3_stage_heatmap.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()  # メモリ解放のためFigureオブジェクトを閉じる
    print(f"保存: {path}")


# Figure 3の描画関数を実行
plot_figure3(df, sample_info, z_data, clusters)